In [34]:
!uv pip install numpy torch torchvision pillow

Audited 4 packages in 20ms


In [35]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [36]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [37]:
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

In [38]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=True, num_workers=2)

In [39]:
image, label = train_data[0]
image.size()

torch.Size([3, 32, 32])

In [40]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [41]:
class NeuralNet(nn.Module):

  def __init__(self):
    super().__init__()

    self.con1 = nn.Conv2d(3, 12, 5) # 32 - 5 = 27 + 1
    self.pool = nn.MaxPool2d(2, 2) # 27 / 2 = 13 + 1
    self.con2 = nn.Conv2d(12, 24, 5) # 14 - 5 = 9 + 1
    self.fc1 = nn.Linear(24 * 5 * 5, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def forward(self, x):
    x = self.pool(F.relu(self.con1(x)))
    x = self.pool(F.relu(self.con2(x)))
    x = torch.flatten(x, 1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)

    return x

In [42]:
net = NeuralNet()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)


In [43]:
for epoch in range(100):
  print(f"Training epoch {epoch + 1}...")

  running_loss = 0.0

  for i, data in enumerate(train_loader):
    inputs, labels = data

    optimizer.zero_grad()

    outputs = net(inputs)

    loss = loss_function(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  print(f"Loss: {running_loss / len(train_loader):.4f}")

print("Finished training")


Training epoch 1...
Loss: 2.1756
Training epoch 2...
Loss: 1.7212
Training epoch 3...
Loss: 1.5150
Training epoch 4...
Loss: 1.4018
Training epoch 5...
Loss: 1.3131
Training epoch 6...
Loss: 1.2301
Training epoch 7...
Loss: 1.1605
Training epoch 8...
Loss: 1.0990
Training epoch 9...
Loss: 1.0448
Training epoch 10...
Loss: 0.9970
Training epoch 11...
Loss: 0.9467
Training epoch 12...
Loss: 0.9097
Training epoch 13...
Loss: 0.8679
Training epoch 14...
Loss: 0.8382
Training epoch 15...
Loss: 0.8019
Training epoch 16...
Loss: 0.7748
Training epoch 17...
Loss: 0.7453
Training epoch 18...
Loss: 0.7172
Training epoch 19...
Loss: 0.6942
Training epoch 20...
Loss: 0.6676
Training epoch 21...
Loss: 0.6399
Training epoch 22...
Loss: 0.6164
Training epoch 23...
Loss: 0.5922
Training epoch 24...
Loss: 0.5722
Training epoch 25...
Loss: 0.5499
Training epoch 26...
Loss: 0.5281
Training epoch 27...
Loss: 0.5073
Training epoch 28...
Loss: 0.4885
Training epoch 29...
Loss: 0.4649
Training epoch 30...
Lo

In [44]:
torch.save(net.state_dict(), 'trained_net.pth')

In [45]:
net = NeuralNet()
net.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [46]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
  for data in test_loader:
    images, labels = data
    outputs = net(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy}%")

Accuracy: 67.65%


In [47]:
new_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def load_image(image_path):
    image = Image.open(image_path)
    image = new_transform(image)
    image = image.unsqueeze(0)
    return image

image_paths = [
    'example1.jpg', 
    'example2.jpg', 
    'example3.jpg'
]
images = [load_image(path) for path in image_paths]

net.eval()
with torch.no_grad():
  for image in images:
    output = net(image)
    _, predicted = torch.max(output, 1)
    print(f"Prediction: {class_names[predicted.item()]}")

Prediction: dog
Prediction: bird
Prediction: plane
